In [1]:
import torch
from pathlib import Path


def resolve_repo_root() -> Path:
    candidates = [
        Path.cwd().resolve(),
        *Path.cwd().resolve().parents,
        Path("/home/snt/projects/AgenticMemoryNav"),
    ]
    seen = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        probe = candidate / "external-lib" / "lingbot-map" / "models" / "lingbot-map"
        if probe.exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repo root. Run this from the project root or a child directory."
    )


def check_point_head(ckpt_path: Path):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dict = ckpt.get("model", ckpt) if isinstance(ckpt, dict) else ckpt

    point_keys = [k for k in state_dict.keys() if "point_head" in k]
    print(f"{ckpt_path}:")
    print(f"  total keys: {len(state_dict)}")
    print(f"  point_head keys: {len(point_keys)}")
    if point_keys:
        print("  examples:", point_keys[:10])
    else:
        print("  -> No point_head.* weights found!")
    print()


if __name__ == "__main__":
    repo_root = resolve_repo_root()
    base = repo_root / "external-lib" / "lingbot-map" / "models" / "lingbot-map"

    for name in ["lingbot-map.pt", "lingbot-map-long.pt", "lingbot-map-stage1.pt"]:
        check_point_head(base / name)


/home/snt/projects/AgenticMemoryNav/external-lib/lingbot-map/models/lingbot-map/lingbot-map.pt:
  total keys: 1342
  point_head keys: 0
  -> No point_head.* weights found!

/home/snt/projects/AgenticMemoryNav/external-lib/lingbot-map/models/lingbot-map/lingbot-map-long.pt:
  total keys: 1342
  point_head keys: 0
  -> No point_head.* weights found!

/home/snt/projects/AgenticMemoryNav/external-lib/lingbot-map/models/lingbot-map/lingbot-map-stage1.pt:
  total keys: 1403
  point_head keys: 62
  examples: ['point_head.norm.weight', 'point_head.norm.bias', 'point_head.projects.0.weight', 'point_head.projects.0.bias', 'point_head.projects.1.weight', 'point_head.projects.1.bias', 'point_head.projects.2.weight', 'point_head.projects.2.bias', 'point_head.projects.3.weight', 'point_head.projects.3.bias']



In [2]:
import torch
from pathlib import Path


def resolve_repo_root() -> Path:
    candidates = [
        Path.cwd().resolve(),
        *Path.cwd().resolve().parents,
        Path("/home/snt/projects/AgenticMemoryNav"),
    ]
    seen = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        probe = candidate / "external-lib" / "lingbot-map" / "models" / "lingbot-map"
        if probe.exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repo root from this notebook."
    )


repo_root = resolve_repo_root()
base = repo_root / "external-lib" / "lingbot-map" / "models" / "lingbot-map"
ckpt_path = base / "lingbot-map.pt"

print("Repo root:", repo_root)
print("Checkpoint path:", ckpt_path)
print("Exists:", ckpt_path.exists())

Repo root: /home/snt/projects/AgenticMemoryNav
Checkpoint path: /home/snt/projects/AgenticMemoryNav/external-lib/lingbot-map/models/lingbot-map/lingbot-map.pt
Exists: True


In [3]:
print("Loading checkpoint from:", ckpt_path)

ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

if isinstance(ckpt, dict):
    state_dict = ckpt.get("model", ckpt)
    config = ckpt.get("config", {})
else:
    state_dict = ckpt
    config = {}

print("Checkpoint top-level keys:", list(ckpt.keys()) if isinstance(ckpt, dict) else type(ckpt))
print("Config (if any):", config)

Loading checkpoint from: /home/snt/projects/AgenticMemoryNav/external-lib/lingbot-map/models/lingbot-map/lingbot-map.pt
Checkpoint top-level keys: ['aggregator.camera_token', 'aggregator.register_token', 'aggregator.scale_token', 'aggregator.patch_embed.cls_token', 'aggregator.patch_embed.pos_embed', 'aggregator.patch_embed.register_tokens', 'aggregator.patch_embed.mask_token', 'aggregator.patch_embed.patch_embed.proj.weight', 'aggregator.patch_embed.patch_embed.proj.bias', 'aggregator.patch_embed.blocks.0.norm1.weight', 'aggregator.patch_embed.blocks.0.norm1.bias', 'aggregator.patch_embed.blocks.0.attn.qkv.weight', 'aggregator.patch_embed.blocks.0.attn.qkv.bias', 'aggregator.patch_embed.blocks.0.attn.proj.weight', 'aggregator.patch_embed.blocks.0.attn.proj.bias', 'aggregator.patch_embed.blocks.0.ls1.gamma', 'aggregator.patch_embed.blocks.0.norm2.weight', 'aggregator.patch_embed.blocks.0.norm2.bias', 'aggregator.patch_embed.blocks.0.mlp.fc1.weight', 'aggregator.patch_embed.blocks.0.mlp

In [4]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
while not (repo_root / "external-lib" / "lingbot-map").exists():
    if repo_root == repo_root.parent:
        raise FileNotFoundError("Could not locate repo root for external-lib/lingbot-map")
    repo_root = repo_root.parent

external_pkg = repo_root / "external-lib" / "lingbot-map"
if str(external_pkg) not in sys.path:
    sys.path.insert(0, str(external_pkg))

try:
    from lingbot_map.models.gct_stream import GCTStream as ModelClass
except ImportError as e:
    print("ImportError:", e)
    print(
        "未能从 external-lib/lingbot-map 中导入模型。\n"
        "请确认包路径是：\n"
        "  external-lib/lingbot-map/lingbot_map/models/gct_stream.py\n"
        "并检查是否有额外依赖未安装。"
    )
    raise

/home/snt/projects/AgenticMemoryNav/.lingbot-venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# 尝试从 config 构造模型（如果 repo 提供 from_config 之类的方法）
if hasattr(ModelClass, "from_config"):
    model = ModelClass.from_config(config)
else:
    # 退而求其次：直接用默认构造
    model = ModelClass()

print("Model class:", type(model).__name__)

# 加载参数（允许 missing/unexpected，因为主要想看结构）
missing, unexpected = model.load_state_dict(state_dict, strict=False)

print("Missing keys (may be OK):", missing[:20], "..." if len(missing) > 20 else "")
print("Unexpected keys (may be OK):", unexpected[:20], "..." if len(unexpected) > 20 else "")

pretrained_path: 


Failed to load pretrained weights: [Errno 2] No such file or directory: ''


Model class: GCTStream
Missing keys (may be OK): [] 
Unexpected keys (may be OK): [] 


In [6]:
print("=" * 80)
print("MODEL ARCHITECTURE (print(model)):")
print("=" * 80)
print(model)

MODEL ARCHITECTURE (print(model)):
GCTStream(
  (aggregator): AggregatorStream(
    (patch_embed): DinoVisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14))
        (norm): Identity()
      )
      (blocks): ModuleList(
        (0-23): 24 x Block(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Attention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (q_norm): Identity()
            (k_norm): Identity()
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): LayerScale()
          (drop_path1): Identity()
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        

In [7]:
print("=" * 80)
print("HEAD-RELATED MODULES AND PARAMETERS")
print("=" * 80)

head_keywords = ["point_head", "depth_head", "camera_head", "head"]

for name, module in model.named_modules():
    if any(k in name for k in head_keywords):
        print(f"\nModule: {name}")
        print(f"  Type: {type(module).__name__}")
        param_names = [n for n, _ in module.named_parameters()]
        print(f"  Parameters ({len(param_names)}):")
        for pn in param_names[:10]:
            print(f"    - {pn}")
        if len(param_names) > 10:
            print(f"    ... ({len(param_names) - 10} more)")

HEAD-RELATED MODULES AND PARAMETERS

Module: aggregator.patch_embed.head
  Type: Identity
  Parameters (0):

Module: camera_head
  Type: CameraCausalHead
  Parameters (69):
    - empty_pose_tokens
    - trunk.0.norm1.weight
    - trunk.0.norm1.bias
    - trunk.0.attn.qkv.weight
    - trunk.0.attn.qkv.bias
    - trunk.0.attn.proj.weight
    - trunk.0.attn.proj.bias
    - trunk.0.ls1.gamma
    - trunk.0.norm2.weight
    - trunk.0.norm2.bias
    ... (59 more)

Module: camera_head.trunk
  Type: Sequential
  Parameters (56):
    - 0.norm1.weight
    - 0.norm1.bias
    - 0.attn.qkv.weight
    - 0.attn.qkv.bias
    - 0.attn.proj.weight
    - 0.attn.proj.bias
    - 0.ls1.gamma
    - 0.norm2.weight
    - 0.norm2.bias
    - 0.mlp.fc1.weight
    ... (46 more)

Module: camera_head.trunk.0
  Type: CameraBlock
  Parameters (14):
    - norm1.weight
    - norm1.bias
    - attn.qkv.weight
    - attn.qkv.bias
    - attn.proj.weight
    - attn.proj.bias
    - ls1.gamma
    - norm2.weight
    - norm2.bias

In [8]:
print("=" * 80)
print("ALL STATE_DICT KEYS CONTAINING 'point_head'")
print("=" * 80)

point_keys = [k for k in state_dict.keys() if "point_head" in k]
print(f"Total point_head keys: {len(point_keys)}")

for k in point_keys[:30]:
    print(" ", k)
if len(point_keys) > 30:
    print(f"  ... ({len(point_keys) - 30} more)")

ALL STATE_DICT KEYS CONTAINING 'point_head'
Total point_head keys: 0


In [9]:
# 如果上面模型构造失败，可以运行这个 cell 单独分析 state_dict

prefixes = {}
for k in state_dict.keys():
    prefix = k.split(".")[0]
    prefixes.setdefault(prefix, 0)
    prefixes[prefix] += 1

print("Top-level prefixes in state_dict:")
for prefix, count in sorted(prefixes.items(), key=lambda x: -x[1]):
    print(f"  {prefix}: {count} keys")

print("\n" + "=" * 80)
print("ALL KEYS CONTAINING 'point_head'")
print("=" * 80)

point_keys = [k for k in state_dict.keys() if "point_head" in k]
print(f"Total point_head keys: {len(point_keys)}")
for k in point_keys[:30]:
    print(" ", k)
if len(point_keys) > 30:
    print(f"  ... ({len(point_keys) - 30} more)")

Top-level prefixes in state_dict:
  aggregator: 1211 keys
  camera_head: 69 keys
  depth_head: 62 keys

ALL KEYS CONTAINING 'point_head'
Total point_head keys: 0
